In [ ]:
import numpy as np
from pathlib import Path
import re
np.random.seed(49)

In [ ]:
# Implement a parser
def parser(pos_folder, neg_folder, label_pos = 1, label_neg = -1, encoding='utf-8'):
    '''Read 2000 txt files from two folders, save raw text from
    each file to X_Raw, and save the relavent label to y. Label "1"
    means positive, "-1" means negative.'''
    X_raw, y = [], []
    pos_folder = Path(pos_folder)
    neg_folder = Path(neg_folder)
    for folder, label in [(pos_folder, label_pos), (neg_folder, label_neg)]:
        for p in sorted(folder.glob("*.txt")):
            text = p.read_text(encoding=encoding)
            X_raw.append(text)
            y.append(label)
    return X_raw, y

pos_folder = "txt_sentoken/pos"
neg_folder = "txt_sentoken/neg"
X_raw, y = parser(pos_folder, neg_folder)
X_raw = np.array(X_raw)
y = np.array(y)

# shuffle the data
N = len(X_raw)
perm = np.random.permutation(N)
X_raw = X_raw[perm]
y = y[perm]

print(len(X_raw), len(y))


In [ ]:
assert len(X_raw) == 2000
assert np.all([isinstance(x, str) for x in X_raw])
assert len(X_raw) == y.shape[0]
assert len(np.unique(y))==2
assert y.min() == -1
assert y.max() == 1

In [ ]:
# Splitting off the test set
split_point = int(len(X_raw)*0.8)
X_train = X_raw[:split_point]
X_test = X_raw[split_point:]
y_train = y[:split_point]
y_test = y[split_point:]

assert len(y_train)+len(y_test) == len(y)
assert len(y_test) > 10

In [ ]:
class BinaryBowVectorizer:

    def __init__(self, min_count):
        self.min_count = min_count
        self.vocab = None
        self.ordered_vocabulary = None

    def tokenizer(self, text:str):
        tokenizer = re.compile(r"\w+(?:['-]\w+)*|[^\w\s]")
        return tokenizer.findall(text)
    
    def filter_tokens(self, tokens):
        keep = ["!", "?"]
        out = []
        for token in tokens:
            if token in keep:
                out.append(token)
            elif re.match(r"\w+(?:[-']\w+)*$", token):
                out.append(token)
            else:
                continue
        return out
    
    def vocabulary(self, sentence, min_count=None):
        '''Take a sentence as an element from a list/array, then build an empty dictionary, then clean the data and save it and its frequency
        as paires in the dictionary.'''
        v = {}
        for words in sentence:
            for word in self.filter_tokens(self.tokenizer(words)):
                if word not in v:
                    v[word] = 1
                else:
                    v[word] += 1
        # Filter tokens appearence less than min_count times, to reduce noise
        vocab = {}
        for token, freq in v.items():
            if freq >= min_count:
                vocab[token] = freq
            
        return vocab
    
    def transform_binary_bow(self, X, vocal):
        '''Take two parameters, X is a list of raw text, vocal is a vocabulary 
        dictionary with key is token and i is index. Here implement a method turn 
        the X to a binary bag-of-words matrix.'''
        V = len(vocal)
        X_mat = np.zeros((len(X), V))
        for i, text in enumerate(X):
            tokens = self.filter_tokens(self.tokenizer(text))
            for token in tokens:
                j = vocal.get(token)
                if j is not None:
                    X_mat[i, j] = 1
        return X_mat

    # sklearn API style:
    def fit(self, X_raw):
        voc = self.vocabulary(X_raw, min_count=self.min_count) # get the vocabulary 
        voc = sorted(voc.items(), key=lambda x: x[1], reverse=True) # sort the vocabulary dictionary with frequency
        ordered_vocabulary = [token for token, _ in voc] # remove frequencies, get a list of sorted tokens
        vocab = {token: i for i, token in enumerate(ordered_vocabulary)} # one token maps one number

        self.ordered_vocabulary = ordered_vocabulary
        self.vocab = vocab
        return self
    
    def transform(self, X_Raw):
        return self.transform_binary_bow(X_Raw, self.vocab)
    
    def fit_transform(self, X_raw):
        return self.fit(X_raw).transform(X_raw)

        

In [ ]:
vec = BinaryBowVectorizer(min_count=3)
X_train = vec.fit_transform(X_train)
X_test  = vec.transform(X_test)

print(X_train[:2, :100])
print(X_test[:2, 100:200])
print(X_train.shape)
print(X_test.shape)

for w in ['dolphin', 'the', 'coffee', 'wo']:
    print(f"'{w}' in vocabulary: {w in vec.vocab}")



In [ ]:
# Implement a linear classifier model with hinge loss to classify the positive and negative data
class HingeLossClassifier:

    def __init__(self, lr, max_iterations, reg_damp):
        self.lr = lr # learning rate
        self.max_iterations = max_iterations
        self.reg_damp = reg_damp 
        self._theta = None
        self.loss_ = []
        

    def decision_function(self, X):
        '''Implement a linear function'''
        w = self._theta[1:]
        b = self._theta[0]
        score = X @ w + b
        return score

    def predict(self, X):
        '''Mapping scores to +1 or -1, positive socores +1, negative scores -1, return an array'''
        scores = self.decision_function(X)
        y_hat = []
        for score in scores:
            if score >= 0:
                y_hat.append(1)
            else:
                y_hat.append(-1)
        return np.array(y_hat)
    
    def _hinge_loss(self, X, y):
        '''Implement hinge loss function. X should be an (N, V) array and y should be (N, ),
        and y only contains -1 and 1.'''
        assert X.shape[0] == y.shape[0]

        scores = self.decision_function(X)
        margins = y * scores
        hinge = np.maximum(0.0, 1.0 - margins)
        loss = np.sum(hinge)
        return loss
    
    def _regularizer(self):
        '''Implement L1 penalty on weights, not including bias'''
        w = self._theta[1:]
        w_abs = np.abs(w)
        l1 = np.sum(w_abs)
        return self.reg_damp * l1
    
    def objective(self, X, y):
        return self._hinge_loss(X, y) + self._regularizer()
    
    def score(self, X, y):
        y_pred = self.predict(X)
        correct = (y_pred == y)
        accuracy = np.mean(correct)
        return accuracy

    def gradient(self, X, y):
        '''Compute gradient of hinge loss. Only samples with margin < 1 contribut to
        hinge loss gradient.'''
        scores = self.decision_function(X)
        margins = y * scores
        active = (margins < 1)
        X_act = X[active] # feature vectors of active samples
        y_act = y[active]

        # implement the derivation of the function
        grad_b = -np.sum(y_act) 
        grad_w = -y_act @ X_act

        grad_theta = np.zeros_like(self._theta, dtype=float)
        grad_theta[0] = grad_b # gradient for bias
        grad_theta[1:] = grad_w # gradient for weights

        # add l1 regularization gradient
        sign_w = np.sign(self._theta[1:])
        reg_grad_w = self.reg_damp * sign_w
        grad_theta[1:] += reg_grad_w
        
        return grad_theta

    def fit(self, X, y):
        V = X.shape[1]
        self._theta = np.zeros(V+1)

        # compute gradient of hinge loss + L1 regularization
        for _ in range(self.max_iterations):
            g = self.gradient(X, y)
            self._theta = self._theta - self.lr * g
            self.loss_.append(self.objective(X, y)) # moniter objective values
        return self
    


In [ ]:
import matplotlib.pyplot as plt

model = HingeLossClassifier(lr=0.0001,max_iterations=1000,reg_damp=0.001)
model.fit(X_train, y_train)
plt.figure(figsize=(20, 3))
plt.plot(model._theta[1:])
plt.xlabel("Value")
plt.ylabel("Weights")
plt.show()

In [ ]:
assert (len(model._theta)-1) == len(vec.ordered_vocabulary)

# Sort by absolute value
idx = np.argsort(np.abs(model._theta[1:]))

print("                Word   Weight  Occurences")
for i in idx[-20:]:   # Pick those with highest 'voting' values
  print("%20s   %.3f\t%i " % (vec.ordered_vocabulary[i], model._theta[i+1], np.sum([vec.ordered_vocabulary[i] in d for d in X_raw])))

In [ ]:
np.random.seed(49)
lr_list = [1e-4, 1e-3, 1e-2, 1e-1]
reg_list = [1e-4, 1e-3, 1e-2, 1e-1]

max_iterations = 200

result = []
# grid search 
for lr in lr_list:
    for reg in reg_list:
        # train a model with different combinations of hyperparameters  
        model = HingeLossClassifier(lr=lr, max_iterations=max_iterations, reg_damp=reg)
        model.fit(X_train, y_train)

        # evaluate on the training set and save the results
        scores = model.score(X_train, y_train)
        result.append((scores, lr, reg))

# sort all results and pick top 5 and the best one
result_sorted = sorted(result, key=lambda x:x[0], reverse=True)
print("Top5 parameter combinations:")
for r in result_sorted[:5]:
    print(r)

best_score, best_lr, best_reg = result_sorted[0]

print("Best parameters:")
print("score:", best_score)
print("lr:", best_lr)
print("reg:", best_reg)


In [ ]:
# Train the model with the best hyperparameters
model_best = HingeLossClassifier(lr=best_lr, max_iterations=200, reg_damp=best_reg)
model_best.fit(X_train, y_train)


In [ ]:
test_accuracy = model_best.score(X_test, y_test)
print("Test accuracy: ", test_accuracy)

# Analysis:

I reimplemented a linear classifier with hinge loss for sentiment polarity classification using binary bag-of-words features. The decision boundary I used is a linear function $f(x) = w^Tx+b$. With binary BoW features, each token contributes its weight to the score. Positive weights push predicitons toward +1 and negative weights toward -1. 

To reduce dimentiality and noise, I filtered tokens with frequency less than or equal to three, which reduced the vocabulary size from more than 40000 to roughly half of that.

On training set, I performed a grid search over the learning rate and regularization strength, using values ranging from 1r-4 to 1e-1 on the training set. The best-performing combination was lr=0.0001 and reg=0.0001. Using these hyperparameters, the model achieved a test accuracy of 0.8825.

# Reflection

For me, the most difficult part of the assignment was understanding the ideas of regularization and related concepts such as normalization, and how they affect the learning process.

The most educational part of the assignment was implementing the full pipeline from raw text to prediction. It helped me understand how parsing, feature extraction, and optimization are connected. The part that took the most time was debugging the model and making sure that the dimensions, margins, and parameter updates were correct.
